In [ ]:
import os
os.environ['GENERICIO_NO_MPI'] = 'true'
# Force trame backend
os.environ['PYVISTA_OFF_SCREEN'] = 'false'
os.environ['PYVISTA_USE_PANEL'] = '0'

import pyvista as pv
import pygio
import numpy as np

# Set trame backend BEFORE any visualization
pv.set_jupyter_backend('trame')
pv.global_theme.jupyter_backend = 'trame'

# Load ALL 8 GenericIO files
base_filename = "/projects/exasky/data/hacc/SCIDAC_RUNS/128MPC_RUNS_FLAMINGO_DESIGN_3A/FSN_0.5387_VEL_149.279_TEXP_9.613_BETA_0.8710_SEED_1.387e5/output/m000p.full.mpicosmo.567"

print("Loading all files...")
all_positions = []
all_masses = []
all_uu = []

# Load the 8 numbered files
for i in range(8):
    filename = f"{base_filename}#{i}"
    print(f"Loading file {i+1}/8: {filename}")
    data = pygio.read_genericio(filename)
    
    positions = np.stack([data['x'], data['y'], data['z']], axis=1)
    all_positions.append(positions)
    all_masses.append(data['mass'])
    all_uu.append(data['uu'])
    
    print(f"  ✓ {len(positions):,} particles")

# Concatenate all data
print("\nCombining all files...")
positions = np.vstack(all_positions)
data = {
    'mass': np.concatenate(all_masses),
    'uu': np.concatenate(all_uu)
}

print(f"\nTOTAL PARTICLES: {len(positions):,}")
print(f"X range: [{positions[:, 0].min():.2f}, {positions[:, 0].max():.2f}]")
print(f"Y range: [{positions[:, 1].min():.2f}, {positions[:, 1].max():.2f}]")
print(f"Z range: [{positions[:, 2].min():.2f}, {positions[:, 2].max():.2f}]")

# Subsample for point cloud
subsample_factor = 30
indices = np.random.choice(len(positions), len(positions)//subsample_factor, replace=False)
points_subsample = positions[indices]
print(f"Subsampled to: {len(points_subsample):,} points")

# CREATE DENSITY GRIDS
grid_size = 128

print("Creating density grids...")

# Particle density grid
hist_density, edges = np.histogramdd(
    positions, 
    bins=grid_size, 
    range=[[0, 128], [0, 128], [0, 128]]
)

# Mass-weighted density
hist_mass, _ = np.histogramdd(
    positions, 
    bins=grid_size,
    range=[[0, 128], [0, 128], [0, 128]],
    weights=data['mass']
)

# Temperature grid
hist_temp, _ = np.histogramdd(
    positions,
    bins=grid_size,
    range=[[0, 128], [0, 128], [0, 128]],
    weights=data['uu']
)
hist_temp = np.divide(hist_temp, hist_density, where=hist_density>0)

print("\n" + "="*50)
print("SPATIAL ALIGNMENT CHECK")
print("="*50)

print(f"\n📍 PARTICLE DATA:")
print(f"  Shape: {positions.shape}")
print(f"  X: [{positions[:, 0].min():.2f}, {positions[:, 0].max():.2f}]")
print(f"  Y: [{positions[:, 1].min():.2f}, {positions[:, 1].max():.2f}]")
print(f"  Z: [{positions[:, 2].min():.2f}, {positions[:, 2].max():.2f}]")

print(f"\n📊 GRID DATA:")
print(f"  Density shape: {hist_density.shape}")
print(f"  Grid bin edges:")
print(f"    X: [{edges[0][0]:.2f}, {edges[0][-1]:.2f}]")
print(f"    Y: [{edges[1][0]:.2f}, {edges[1][-1]:.2f}]")
print(f"    Z: [{edges[2][0]:.2f}, {edges[2][-1]:.2f}]")

in_bounds = np.all(
    (positions >= [edges[0][0], edges[1][0], edges[2][0]]) &
    (positions <= [edges[0][-1], edges[1][-1], edges[2][-1]]),
    axis=1
)
print(f"\n✓ Particles inside grid: {in_bounds.sum():,} / {len(positions):,} ({100*in_bounds.mean():.1f}%)")
print("="*50 + "\n")

# ===== PYVISTA VISUALIZATION =====
print("Creating PyVista visualization...")

# Create plotter with notebook mode
pl = pv.Plotter(notebook=True)

# 1. Add volume rendering for density (log scale)
grid_density = pv.ImageData(dimensions=hist_density.shape)
grid_density.point_data['density'] = np.log10(hist_density.flatten(order='F') + 1)
grid_density.origin = (0, 0, 0)
grid_density.spacing = (128/grid_size, 128/grid_size, 128/grid_size)

pl.add_volume(
    grid_density,
    scalars='density',
    cmap='viridis',
    opacity='sigmoid',
    name='Particle Density (log)'
)

# 2. Add mass density volume (optional - comment out if too slow)
# grid_mass = pv.ImageData(dimensions=hist_mass.shape)
# grid_mass.point_data['mass'] = np.log10(hist_mass.flatten(order='F') + 1)
# grid_mass.origin = (0, 0, 0)
# grid_mass.spacing = (128/grid_size, 128/grid_size, 128/grid_size)
# pl.add_volume(grid_mass, scalars='mass', cmap='inferno', opacity='sigmoid')

# 3. Add point cloud
point_cloud = pv.PolyData(points_subsample)
point_cloud['mass'] = data['mass'][indices]
point_cloud['uu'] = data['uu'][indices]

pl.add_mesh(
    point_cloud,
    scalars='mass',
    cmap='plasma',
    point_size=3,
    render_points_as_spheres=True,
    opacity=0.5,
    name='Particles'
)

# Camera and styling
pl.camera_position = 'iso'
pl.add_axes()
pl.add_bounding_box()

print("Visualization ready! Showing interactive viewer...")
pl.show(jupyter_backend='trame')

Widget(value='<iframe src="http://localhost:46811/index.html?ui=P_0x1490fbc7ac60_1&reconnect=auto" class="pyvi…

In [ ]:
print('hi')